# TiRex: xLSTM возвращается

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/24_tirex.ipynb)

**Внимание:** TiRex требует GPU с compute capability 8.0+ (Ampere и новее) для полной скорости.

## Установка зависимостей

In [ ]:
# TiRex требует специфичных зависимостей
!pip install -q torch pandas numpy matplotlib
# Для установки TiRex следуйте инструкциям в репозитории NX-AI/TiRex

## Подготовка данных

In [ ]:
import torch
import pandas as pd
import numpy as np

# Создаём синтетические данные
np.random.seed(42)

# Параметры
n_series = 10
context_length = 128
horizon = 32

# Генерируем данные
data = []
for i in range(n_series):
    t = np.arange(context_length)
    y = 100 + np.cumsum(np.random.randn(context_length)) + 20 * np.sin(t / 7 * 2 * np.pi)
    data.append(y)

context = torch.tensor(np.array(data), dtype=torch.float32)
print(f"Context shape: {context.shape}")

## TiRex: прогнозирование (концептуальный код)

**Примечание:** Полная работа с TiRex требует установки из официального репозитория.

In [ ]:
# Концептуальный код работы с TiRex
# В реальности требуется установка из https://github.com/NX-AI/TiRex

'''
from tirex import load_model, ForecastModel

# Загрузка предобученной модели
model: ForecastModel = load_model("NX-AI/TiRex")
model.to('cuda')

# Подготовка данных
# TiRex ожидает тензор (batch, time)
data = context.to('cuda')

# Прогнозирование
quantiles, mean = model.forecast(
    context=data,
    prediction_length=horizon
)

# quantiles: (batch, 9, horizon) — 9 квантилей
# mean: (batch, horizon) — среднее (медиана)

print(f"Прогноз shape: {mean.shape}")
print(f"Квантили shape: {quantiles.shape}")

# Доступ к конкретным квантилям
# Индексы: 0=0.1, 1=0.2, ..., 4=0.5 (медиана), ..., 8=0.9
p10 = quantiles[:, 0, :]  # 10-й перцентиль
p50 = quantiles[:, 4, :]  # медиана
p90 = quantiles[:, 8, :]  # 90-й перцентиль
'''

print("TiRex требует установки из официального репозитория NX-AI/TiRex")

## Визуализация вероятностного прогноза (шаблон)

In [ ]:
import matplotlib.pyplot as plt

def plot_tirex_forecast(history, quantiles, mean, title=''):
    """
    Визуализация прогноза TiRex с квантилями.
    
    Args:
        history: исторические значения (1D array)
        quantiles: квантили прогноза (9, horizon)
        mean: средний прогноз (horizon,)
    """
    fig, ax = plt.subplots(figsize=(12, 4))
    
    # История
    ax.plot(range(len(history)), history, 
            color='black', label='История', linewidth=1.5)
    
    # Прогноз
    forecast_start = len(history)
    forecast_range = range(forecast_start, forecast_start + len(mean))
    
    # Медиана
    ax.plot(forecast_range, mean, 
            color='blue', label='Медиана', linewidth=2)
    
    # 80% интервал (p10-p90)
    ax.fill_between(
        forecast_range,
        quantiles[0],  # p10
        quantiles[8],  # p90
        alpha=0.2, color='blue', label='80% интервал'
    )
    
    # 50% интервал (p30-p70)
    ax.fill_between(
        forecast_range,
        quantiles[2],  # p30
        quantiles[6],  # p70
        alpha=0.4, color='blue', label='50% интервал'
    )
    
    ax.axvline(x=forecast_start, color='gray', linestyle='--', alpha=0.5)
    ax.legend()
    ax.set_title(title)
    ax.set_xlabel('Время')
    ax.set_ylabel('Значение')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Демонстрация с синтетическими данными
sample_history = context[0].numpy()

# Имитация прогноза
np.random.seed(42)
sample_mean = sample_history[-1] + np.cumsum(np.random.randn(horizon) * 2)
sample_quantiles = np.array([
    sample_mean - 3 * (i + 1) for i in range(5)
][::-1] + [
    sample_mean + 3 * (i + 1) for i in range(4)
])

plot_tirex_forecast(sample_history, sample_quantiles, sample_mean, 
                    title='TiRex: пример прогноза с квантилями')

## Подготовка данных из pandas DataFrame

In [ ]:
def prepare_data_for_tirex(df, context_length, unique_id_col='unique_id', value_col='y'):
    """Преобразует DataFrame в тензоры для TiRex."""
    series_list = []
    ids = df[unique_id_col].unique()
    
    for uid in ids:
        series = df[df[unique_id_col] == uid][value_col].values
        # Берём последние context_length точек
        if len(series) >= context_length:
            series_list.append(series[-context_length:])
    
    return torch.tensor(np.array(series_list), dtype=torch.float32)

# Пример с DataFrame
dates = pd.date_range('2023-01-01', periods=200, freq='D')
df = pd.DataFrame({
    'unique_id': np.repeat([f'series_{i}' for i in range(5)], 200),
    'ds': np.tile(dates, 5),
    'y': np.random.randn(1000).cumsum() + 100
})

context_tensor = prepare_data_for_tirex(df, context_length=128)
print(f"Подготовлено рядов: {context_tensor.shape[0]}")
print(f"Длина контекста: {context_tensor.shape[1]}")